<a href="https://colab.research.google.com/github/isismeira/natural_language_processing/blob/main/classificacao_neural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
cd drive/MyDrive/corona_tweets_nlp/

In [ ]:
import torch
import torchtext
from torchtext import data
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class TextSentiment(nn.Module):
  def __init__(self, vocab_size, embed_dim, drop_prob, hs1, num_class,):
    super().__init__()
    self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=True)
    self.relu = nn.ReLU()
    self.softmax = nn.Softmax(dim=1)
    self.dropout = nn.Dropout(drop_prob)
    self.fc1 = nn.Linear(embed_dim, hs1)
    self.fc2 = nn.Linear(hs1, num_class)
    self.init_weights()


  def init_weights(self):
    initrange = 0.5
    self.embedding.weight.data.uniform_(-initrange, initrange)
    self.fc1.weight.data.uniform_(-initrange, initrange)
    self.fc2.weight.data.uniform_(-initrange, initrange)
    self.fc1.bias.data.zero_()
    self.fc2.bias.data.zero_()

  def forward(self, text, offsets):
    embedded = self.embedding(text,offsets)
    x = embedded.view(embedded.shape[0], -1)
    x = self.dropout(x)
    preds = self.softmax(self, fc2(x))
    return preds

In [ ]:
text = data.Field(tokenize='spacy')

train_data = data.TabularDataset(path="Corona_NLP_train.csv",
                  format="csv",
                  fields=[
                      ('text', text),
                      ('target', data.Field())],
                  skip_header=True)

test_data = data.TabularDataset(path="Corona_NLP_test.csv",
                  format="csv",
                  fields=[
                      ('text', text),
                      ('target', data.Field())],
                  skip_header=True)

text.build_vocab(train_data, test_data)

In [ ]:
VOCAB_SIZE = len(text.vocab)
BATCH_SIZE = 8
EMBED_DIM = 32
NUM_CLASS = 5
DROPOUT_PROB = 0.5
HS1 = 128
model = TextSentiment(VOCAB_SIZE, EMBED_DIM, DROPOUT_PROB, HS1, NUM_CLASS)

In [ ]:
def generate_batch(batch):
  label = torch.tensor([int(entry.target[0]) for entry in batch])
  _text = []
  for entry in batch:
    _entry = []
    for t in entry.text:
      entry.append(text.vocab.stoi[t])
    _text.append(torch.tensor(_entry,dtype=torch.long))
  offsets = [0] + [len(entry) for entry in _text]
  offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
  _text = torch.cat(_text)
  return _text, offsets, label


In [ ]:
from torch.utils.data import DataLoader

def train_func(sub_train_)
  train_loss = 0
  train_acc = 0
  data = DataLoader(sub_train_, batch_size=BATCH_SIZE, shuffle=True, collate_fn=generate_batch)

  for i, (text, offsets, cls) in enumerate(data):
    optmizer.zero_grad()
    text, offsets, cls = text.to(device), offsets.to(device), cls.to(device)
    output = model(text, offsets)
    loss = criterion(output, cls)
    train_loss += loss.item()
    optmizer.zero_grad()
    loss.backward()
    optmizer.step()
    train_acc += (output.argmax(1) == cls).sum().item()

 scheduler.step()

 return train_loss / len(sub_train_), train_acc / len(sub_train_)


def test(data_):
  loss = 0
  acc = 0
  data = DataLoader(data_, batch_size=BATCH_SIZE, collate_fn=generate_batch)
  for text, offsets, cls in data:
    text, offsets, cls = text.to(device), offsets.to(device), cls.to(device)
    with torch.no_grad():
      output = model(text, offsets)
      loss = criterion(output, cls)
      loss += loss.item()
      acc += (output.argmax(1) == cls).sum().item()

  return loss / len(data_), acc / len(data_)

In [ ]:
import time
from torch.utils.data.dataset import random_split
import torch.optim as optim

N_EPOCHS = 20
min_valid_loss = float('inf')

criterion =  torch.nn.CrossEntropyLoss().to(device)
optmizer = torch.optim.SGD(model.parameters(), lr=1.0)
scheduler = torch.optim.lr_scheduler.StepLR(optmizer, 1, gamma=0.9)

train_len = int(len(train_data) * 0.95)
sub_train_, sub_valid_ = random_split(train_data, [train_len, len(train_data) - train_len])
sub_train_

In [ ]:
for epoch in range(N_EPOCHS):
  start_time = time.time()
  train_loss, train_acc = train_func(sub_train_)
  valid_loss, valid_acc = test(sub_valid_)

  secs = int(time.time() - start_time)
  mins = secs / 60
  secs = secs % 60

  print(f'Epoch: {epoch + 1}, Time: {mins}m, {secs}s')
  print(f'Loss: {train_loss:.4f}(train)\t|\tAcc: {train_acc * 100:.1f}%(train)')
  print(f'Loss: {valid_loss:.4f}(valid)\t|\tAcc: {valid_acc * 100:.1f}%(valid)')

In [ ]:
print('Checking the results of test dataset...')
test_loss, test_acc = test(test_data)
print(f'Loss: {test_loss:.4f}(test)\t|\tAcc: {test_acc * 100:.1f}%(test)')

In [ ]:
def ngrams_iterator(token_list, ngrams):
  def _get_ngrams(n):
    return zip(*[token_list[i:] for i in range(n)])
  for x in token_list:
    yield x
  for n in range(2, ngrams + 1):
    for x in _get_ngrams(n):
      yield ' '.join(x)

In [ ]:
NGRAMS = 2
label = {
    0: "---",
    1: "-",
    2: "N",
    3: "+",
    4: "+++"
}

def predict(_text, model, vocab, ngrams):
  if len(_text) == 0:
    return 0
  with torch.no_grad():
    _text = [vocab.stoi[token] for token in ngrams_iterator(_text, ngrams)]
    output = model(torch.tensor(_text), torch.tensor([0]))
    return output.argmax(1).item()


model = model.to('cpu'
for entry in test_data[:10]):
  print('Real: ', label[int(entry.target[0])], " ".join(entry.text), label[predict(entry.text, model, text.vocab, NGRAMS)])